In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2025, 12, 29, 00, 1))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 20, 00))

In [3]:
from SDRUtils.products import USD_SOFR_SwapProduct

product = USD_SOFR_SwapProduct()
df = product.build_classification_dataframe(start=start, end=end, cache_path=cache_path, detect_fly=True, detect_curve=True, detect_mms=True)

df

Classifying Trades: 100%|██████████| 2121/2121 [00:11<00:00, 187.55it/s]


,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,package_id,UPI Underlier Name,Platform identifier,Cleared,package_legs,matched_ust_maturity,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date
0,1572688072000000301,2025-12-29 05:10:11+00:00,2026-01-20,2036-01-20 00:00:00,OIS_SWAP,10.15,10Y,True,0.061111,3W,...,None,USD-SOFR-COMPOUND,BILT,I,None,False,NaN,NaN,NaN,2036-01-20
1,1572660520000000301,2025-12-29 05:10:11+00:00,2026-01-20,2036-01-20 00:00:00,OIS_SWAP,10.15,10Y,True,0.061111,3W,...,None,USD-SOFR-COMPOUND,BILT,I,None,False,NaN,NaN,NaN,2036-01-20
2,1563647051000000101,2025-12-29 05:11:56+00:00,2025-12-31,2026-12-28 00:00:00,OIS_SWAP,1.005556,1Y,False,0.005556,spot,...,None,USD-SOFR-COMPOUND,BMTF,I,None,False,NaN,NaN,NaN,2026-12-28
3,1563670963000000101,2025-12-29 05:19:44+00:00,2025-12-31,2026-03-30 00:00:00,OIS_SWAP,0.247222,3M,False,0.005556,spot,...,None,USD-SOFR-OIS Compound,TWSF,I,None,False,NaN,NaN,NaN,2026-03-30
4,1563737164000000101,2025-12-29 05:39:58+00:00,2025-12-31,2045-12-31 00:00:00,OIS_SWAP,20.286111,20Y,False,0.005556,spot,...,None,USD-SOFR-COMPOUND,TWSF,I,None,False,NaN,NaN,NaN,2045-12-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1806,1573656731000000101,2025-12-29 21:54:42+00:00,2025-12-31,2026-12-31 00:00:00,OIS_SWAP,1.013889,1Y,False,0.005556,spot,...,SPREADOVER_1573656731000000101,USD-SOFR-COMPOUND,BBSF,I,[1573656731000000101],True,91282CME8,2-Year,2024-12-31,2026-12-31
1807,1573691775000000101,2025-12-29 21:59:26+00:00,2025-12-31,2032-12-31 00:00:00,OIS_SWAP,7.102778,7Y,False,0.005556,spot,...,SPREADOVER_1573691775000000101,USD-SOFR-COMPOUND,TWSF,I,[1573691775000000101],True,91282CPQ8,7-Year,2025-12-31,2032-12-31
1808,1573723141000000201,2025-12-29 21:52:38+00:00,2025-12-31,2055-11-15 00:00:00,OIS_SWAP,30.308333,30Y,False,0.005556,spot,...,SPREADOVER_1573723141000000201,USD-SOFR-OIS Compound,BILT,I,[1573723141000000201],True,912810UP1,30-Year,2025-11-17,2055-11-15
1809,1573723140000000101,2025-12-29 21:56:25+00:00,2025-12-31,2035-11-15 00:00:00,OIS_SWAP,10.016667,10Y,False,0.005556,spot,...,SPREADOVER_1573723140000000101,USD-SOFR-OIS Compound,BILT,I,[1573723140000000101],True,91282CPJ4,10-Year,2025-11-17,2035-11-15


In [4]:
df.to_csv("sdr_formatted_usd_rates.csv", index=False)

In [7]:
# df[df["package_type"] == "FLY"]

In [9]:
df["trade_label"].value_counts().head(50)

trade_label
spot 10Y                                     162
spot 5Y                                      109
spot 2Y                                       94
spot 30Y                                      79
IMM_H2026 IMM_H2036                           59
IMM_H2026 IMM_H2031                           50
spot 1M                                       40
spot 1Y                                       38
spot 3Y                                       31
IMM_H2026 IMM_H2056                           31
spot 20Y                                      31
spot 3M                                       31
IMM_H2026 IMM_H2028                           30
5Y / 30Y                                      28
spot 7Y                                       22
spot 4Y                                       20
3M 7Y                                         18
IMM_H2026 IMM_H2033                           18
IMM_H2026 IMM_H2029 / IMM_H2026 IMM_H2031     17
10Y / 30Y                                     17
IMM_Z202

In [47]:
pd.to_datetime(raw_df["Execution Timestamp"]).dt.date.value_counts().index[0]

datetime.date(2025, 12, 29)

In [34]:
with_pkg_df.tail(1).to_dict(orient="records")

[{'trade_id': 1574395035000000101,
  'execution_timestamp': Timestamp('2025-12-30 00:32:12+0000', tz='UTC'),
  'effective_date': Timestamp('2026-04-03 00:00:00'),
  'expiration_date': Timestamp('2030-05-31 00:00:00'),
  'product_type': 'OIS_SWAP',
  'tenor_years': 4.219444444444444,
  'tenor_label': '4Y',
  'is_forward': True,
  'forward_start_years': 0.2611111111111111,
  'forward_label': '3M',
  'trade_label': '3M 4Y',
  'notional': 82000000.0,
  'notional_currency': 'USD',
  'fixed_rate': 0.033477,
  'strike': nan,
  'estimated_pv01': 31438.779832216656,
  'package_type': 'SPREADOVER',
  'package_id': 'SPREADOVER_1574395035000000101',
  'package_legs': [1574395035000000101],
  'matched_ust_maturity': True,
  'ust_cusip': '91282CNG2',
  'ust_oi': '5-Year',
  'ust_issue_date': datetime.date(2025, 6, 2),
  'swap_maturity_date': datetime.date(2030, 5, 31)}]

In [37]:
# with_pkg_df["estimated_pv01"].sort_values(key=lambda x: float(str(x).split("/")[0]) if type(x) == str else float(x))

# with_pkg_df["pv01_clean"] = with_pkg_df["estimated_pv01"].apply(lambda x: float(str(x).split("/")[0]) if type(x) == str else float(x))
# with_pkg_df.sort_values(by="pv01_clean", ascending=False).head(10)